# The Eras Tour of Computer Vision
## Era 2: The Neural Nets Era (Workshop Edition)

Now you'll run a pretrained neural network -- one that learned to recognize roughly a
thousand object categories from about a million training images, none of which you provided.

Everything here already runs. There's exactly **one line** for you to tweak: `KEYWORDS`,
just below. Try a few settings and watch precision and recall trade off against each other.

In [ ]:
!git clone -q https://github.com/LinoNU/eras-tour-workshop.git 2>/dev/null || echo 'Already cloned'
%cd eras-tour-workshop
!pip install -q transformers torch pillow

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline

DATA_DIR = Path('data')
labels_df = pd.read_csv(DATA_DIR / 'labels.csv')
classifier = pipeline(task='image-classification', model='google/vit-base-patch16-224')
print(f"Loaded {len(labels_df)} images. Classifier ready.")

## The one knob: `KEYWORDS`

The classifier returns a ranked list of labels with confidence scores, not a simple yes/no.
`KEYWORDS` decides which of those labels count as "mug-like." Try:
- `('coffee mug',)` -- narrow, high precision, might miss some real mugs
- `('mug', 'cup')` -- broader, catches more real mugs, but also more false alarms
- `('mug', 'cup', 'pitcher')` -- broader still

In [ ]:
KEYWORDS = ('coffee mug',)  # TRY CHANGING ME -- e.g. ('mug', 'cup')
TOP_K = 3


def load_image(rel_path: str) -> Image.Image:
    return Image.open(DATA_DIR / rel_path).convert('RGB')


def predict_has_mug(predictions: list, keywords=KEYWORDS, top_k: int = TOP_K) -> dict:
    for pred in predictions[:top_k]:
        if any(k in pred['label'].lower() for k in keywords):
            return {'prediction': True, 'matched_label': pred['label'], 'matched_score': pred['score']}
    return {'prediction': False, 'matched_label': None, 'matched_score': None}

## Try it on a few images

In [ ]:
test_images = [
    'mugs/ambiguous_01.jpg',           # a mug repurposed as a pen holder
    'distractors/travel_tumbler_01.jpg',  # not a mug -- but close
    'mugs/texture_context_06.jpg',     # backlit, tricky lighting
]

fig, axes = plt.subplots(1, len(test_images), figsize=(5 * len(test_images), 5))
for ax, rel_path in zip(axes, test_images):
    image = load_image(rel_path)
    raw_predictions = classifier(image, top_k=TOP_K)
    decision = predict_has_mug(raw_predictions)
    ax.imshow(image)
    ax.set_title(f"{rel_path}\npredicted={decision['prediction']}  label={decision['matched_label']}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Discussion

- Did broadening `KEYWORDS` fix the travel tumbler, or just make it worse?
- Notice this classifier can't point to *where* the mug is -- only guess what the photo is
  mostly about. That's a real limitation, not a bug you can tune away.

One era down to go. This next one is the only one that can explain *why* it thinks what it
thinks.